In [373]:
import pandas as pd
import numpy as np
import os
import datetime as dt
import missingno as msno
import seaborn as sns
import matplotlib as plt

### Data frame creator

In [508]:
df_polish = pd.read_csv('../data/smog_polishraw.csv', encoding='utf-8-sig')
#df_smog = pd.read_csv('../data/smog_raw.csv', encoding='utf-8-sig')
df_smog = pd.read_csv('../data/smog_polishraw.csv', encoding='utf-8-sig')
df_smog2 = pd.read_csv('../data/smog_raw2.csv', encoding='utf-8-sig')

### Pokazanie czy działa formatowanie - brak błędów polskich znaków - niestety nie potrafie naprawić

In [ ]:
print(df_smog['NAME'].head())

### Nazwy nagłówków

In [ ]:
print(df_smog.columns)

### Opis danych [wszystkie dane wypisane jako bool]

In [ ]:
df_smog.isna().info()

### Opis typu danych w kolumnach

In [ ]:
print(df_smog.dtypes)

### Zmiana typu danych w kolumnie 'Date' na datetime

In [515]:
df_smog['TIMESTAMP_DATETIME'] = pd.to_datetime(df_smog['TIMESTAMP'], errors='coerce')

### Wyciągam datę

In [516]:
df_smog['TYLKO_DATA'] = df_smog['TIMESTAMP_DATETIME'].dt.date

### Wyciągamy sam czas (Format: GG:MM:SS)

In [517]:
df_smog['TYLKO_CZAS'] = df_smog['TIMESTAMP_DATETIME'].dt.time

### Usuwamy kolumnę pomocniczą

In [518]:
df_smog = df_smog.drop(columns=['TIMESTAMP_DATETIME'])

### Podgląd danych po zmianie formatowania daty i czasu

In [ ]:
print(df_smog['TYLKO_CZAS'].head())

### Podgląd nazw nagłówków w tabeli

In [ ]:
df_smog.head(2)

### Podstawowe statystyki opisowe [według typów danych]

In [ ]:
df_smog.describe()

### Sortowanie po kodzie pocztowym [TEMPERATURE_AVG to średnia temperatury, kod pocztowy to POST_CODE ;)]

In [ ]:
df_smog.sort_values(['TEMPERATURE_AVG']).head(10)

### Mapa braków

In [ ]:
sns.heatmap(df_smog.isna())

### Set variables

In [524]:
area_dict = {
    "00-09": "mazowieckie",
    "10-19": "warminsko-mazurskie",
    "20-29": "lubelskie",
    "30-39": "malopolskie",
    "50-59": "dolnoslaskie",
    "60-69": "wielkopolskie",
    "70-79": "zachodniopomorskie",
    "80-89": "pomorskie",
    "90-99": "lodzkie"
}
smog_columns = ['LONGITUDE', 'LATITUDE', 'HUMIDITY_AVG', 'PRESSURE_AVG', 'TEMPERATURE_AVG', 'PM10_AVG', 'PM25_AVG']

### Insert area column and get postal code area number

In [525]:
df_smog.insert(2, 'AREA', df_smog['POST_CODE'].astype(str).str[:2].astype(int))

### Data frame shape

In [ ]:
df_smog.shape

In [ ]:
df_smog2.shape

### Data frame first look

In [ ]:
df_smog.describe

### Format float values with precission to 1 [ Convert values to float]

In [ ]:
for x in smog_columns:
    for z, y in df_smog[f"{x}"].items():
        y = float(y)
    #df_smog[f"{x}"] = float(df_smog[f"{x}"]).round(1)    
    #df_smog2[f"{x}"] = df_smog2[f"{x}"].round(1)

### Strip '-' from post code

In [530]:
for x in df_smog['POST_CODE']:
    df_smog['POST_CODE'] = df_smog['POST_CODE'].str.replace("-", "")
    df_smog2['POST_CODE'] = df_smog2['POST_CODE'].str.replace("-", "")

In [ ]:
df_smog.head(5)

In [ ]:
df_smog2.tail(5)

### Shape after first format

In [ ]:
df_smog.shape

In [ ]:
df_smog2.shape

### -[] Set missing values to 0 in smog_columns 

In [ ]:
for x in smog_columns:
    if df_smog[f"{x}"].any() == 0.0:
        print(df_smog[f"{x}"])
        #print(df_smog2[f"{x}"])
        #df_smog[f"{x}"] = df_smog2[f"{x}"]
    #print(df_smog[f"{x}"].isna().sum())

### Check, if every area is in place (longitude and latitude are in Polish area)

In [ ]:
# Define a function and area latitude and longitude min and max values
for y, x in df_smog['LATITUDE'].items() and df_smog['LONGITUDE'].items():
    x = float(x)

def check_areas(df):
    area_values = {
        "Latitude.North < 54.835563": df_smog['LATITUDE'] < 54.8,
        "Latitude.South > 49.002063": df_smog['LATITUDE'] > 49.0,
        "Longitude.East < 24.145562": df_smog['LONGITUDE'] < 24.2,
        "Longitude.West > 14.124562": df_smog['LONGITUDE'] > 14.0
        }
    return area_values

# Define an object table with the area checker

areas = check_areas(df_smog)

# Print if there are any differences in the areas

for rule, result in areas.items():
    print(f"{rule}: {not result.all()}")

### Check, how many of areas have wrong latitudes\longitudes

In [ ]:
# Check, how many differences are in areas

differences = {rule: ~result for rule, result in areas.items()}
summary = {rule: result.sum() for rule, result in differences.items()}

# Print the number of differences

for rule, count in summary.items():
    print(f"{rule}: {count} differences")

### Save dataframe to csv file **Always put as a last cell !**

In [ ]:
ts = dt.datetime.now().strftime("%Y%m%d_%H-%M")
#df_smog.to_csv(f"../tests/smog_raw{ts}.csv", index=False)
#df_smog2.to_csv(f"../tests/smog2_raw{ts}.csv", index=False)
#df_polish.to_csv(f"../tests/smog_polishraw{ts}.csv", index=False, encoding='utf-8-sig')